# Assemble a versioned prepared dataset

Combine an immutable processed-corpus bundle, Wubi mapping, and a compatible pinned acoustic bundle. Run the local corpus-processing section first; assembly copies accepted rows without normalization, filtering, deduplication, or sampling.


## Project imports

Locate the local project and import the prepared-dataset SDK and checksum helper.


In [ ]:
from pathlib import Path
import sys

def find_project_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'datasets').is_dir() and (candidate / 'src').is_dir():
            return candidate
    raise RuntimeError('Could not find a project root containing datasets/ and src/')

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from phonetics import assemble_dataset
from phonetics.artifacts import sha256_file


## Assembly configuration

Pin the processed corpus, Wubi mapping, and acoustic bundle manifests. Compatibility uses the exact Wubi SHA and sound-inventory artifact SHA, so a different acoustic model may be used when its inventory is identical.


In [ ]:
RUN_PREPARATION = True
PROCESSED_CORPUS_PATH = PROJECT_ROOT / 'datasets/processed-corpus/duyu-chinese-lessons-v1'
PROCESSED_CORPUS_MANIFEST_SHA256 = sha256_file(PROCESSED_CORPUS_PATH / 'manifest.json') if RUN_PREPARATION else None
ACOUSTIC_BUNDLE_PATH = PROJECT_ROOT / 'datasets/pinyin-embeddings/duyu-chinese-lessons-zh-cn-v1'
ACOUSTIC_MANIFEST_SHA256 = '25aef7ef70ec8687dda32afa92887130cd815d835f7b4ff6b9344822eb412d11'

CONFIG = {
    'dataset_id': 'duyu-chinese-lessons-zh-cn-v1',
    'output_root': str(PROJECT_ROOT / 'datasets/prepared'),
    'processed_corpus': {'path': str(PROCESSED_CORPUS_PATH), 'manifest_sha256': PROCESSED_CORPUS_MANIFEST_SHA256},
    'wubi': {'path': str(PROJECT_ROOT / 'datasets/wubi/chinese_to_wubi.pkl'), 'sha256': '80d3228546097490632cfdc1b42666b706840e2d7f1f76f64712449c9cfe927e'},
    'acoustic_bundle': {'path': str(ACOUSTIC_BUNDLE_PATH), 'manifest_sha256': ACOUSTIC_MANIFEST_SHA256},
}


## Publish the prepared dataset

Enable the guarded assembly only after pinning the processed-corpus manifest. Existing datasets are never overwritten.


In [ ]:
if RUN_PREPARATION:
    assert "REPLACE_WITH" not in repr(CONFIG)
    prepared_path = assemble_dataset(CONFIG)
    print(prepared_path)
else:
    print("Preparation disabled; review local paths and pinned hashes first.")
